In [1]:
import torch
import matplotlib.pyplot as plt
import random

/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Callbacks

## As a GUI event

In [2]:
import ipywidgets as widgets

From the [ipywidget docs](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Events.html):

- *the button widget is used to handle mouse clicks. The on_click method of the Button can be used to register function to be called when the button is clicked*`

In [3]:
w = widgets.Button(description='click me here')

In [4]:
w

Button(description='click me here', style=ButtonStyle())

In [5]:
def f(o): print('hello')

In [6]:
w.on_click(f)

In [7]:
w

Button(description='click me here', style=ButtonStyle())

- NB: When callbacks are used in this way they are often called "events".

In [8]:
from time import sleep

In [9]:
def slow_calculation():
    res = 0
    for i in range(5):
        res += i*i
        sleep(1)
    return res

In [10]:
slow_calculation()

30

In [11]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        res += i*i
        sleep(1)
        if cb:cb(i)
    return res

In [12]:
def show_progress(epoch): print(f"Awesome! We've finished epoch {epoch}!")

In [13]:
slow_calculation(show_progress)

Awesome! We've finished epoch 0!
Awesome! We've finished epoch 1!
Awesome! We've finished epoch 2!
Awesome! We've finished epoch 3!
Awesome! We've finished epoch 4!


30

## Lambdas and partials

In [14]:
slow_calculation(lambda o : print(f"Great, we are at the {o} epoch "))

Great we are at the 0 epoch 
Great we are at the 1 epoch 
Great we are at the 2 epoch 
Great we are at the 3 epoch 
Great we are at the 4 epoch 


30

In [15]:
def show_progress(exclamation, epoch): print(f"{exclamation}! We've finished epoch {epoch}!")

In [16]:
slow_calculation(lambda o : show_progress("YEAH", o ))

YEAH! We've finished epoch 0!
YEAH! We've finished epoch 1!
YEAH! We've finished epoch 2!
YEAH! We've finished epoch 3!
YEAH! We've finished epoch 4!


30

In [17]:
def make_show_progress(exclamation):
    def _inner(epoch): print(f"{exclamation}! We've finished epoch {epoch}!")
    return _inner

In [18]:
slow_calculation(make_show_progress("Nice!"))

Nice!! We've finished epoch 0!
Nice!! We've finished epoch 1!
Nice!! We've finished epoch 2!
Nice!! We've finished epoch 3!
Nice!! We've finished epoch 4!


30

In [19]:
from functools import partial

`partial`:

You have a function that takes 2 arguments, but you want to call it later with only 1 argument. `partial` locks in one argument ahead of time.

In [20]:
slow_calculation(partial(show_progress, "OK I guess"))

OK I guess! We've finished epoch 0!
OK I guess! We've finished epoch 1!
OK I guess! We've finished epoch 2!
OK I guess! We've finished epoch 3!
OK I guess! We've finished epoch 4!


30

In [21]:
f2 = partial(show_progress, "OK I guess")

## Callbacks as callable classes

In [22]:
class ProgressShowingCallback():
    def __init__(self, exclamation="Awesome"): self.exclamation = exclamation
    def __call__(self, epoch): print(f"{self.exclamation}! We've finished epoch {epoch}!")

In [23]:
cb = ProgressShowingCallback("Just super")

In [24]:
slow_calculation(cb)

Just super! We've finished epoch 0!
Just super! We've finished epoch 1!
Just super! We've finished epoch 2!
Just super! We've finished epoch 3!
Just super! We've finished epoch 4!


30

## Multiple callback funcs; `*args` and `**kwargs`

In [25]:
def f(*a, **b): print(f"args: {a}; kwargs: {b}")

In [26]:
f(3, 'a', thing1="hello")

args: (3, 'a'); kwargs: {'thing1': 'hello'}


In [27]:
def g(a,b,c=0): print(a,b,c)

In [28]:
args = [1,2]
kwargs = {'c':3} # so kwargs is always a dictionary
g(*args, **kwargs)

1 2 3


In [29]:
class PrintStepCallback():
    def before_calc(self, *args, **kwargs): print(f"About to start")
    def after_calc (self, *args, **kwargs): print(f"Done step")

In [30]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        if cb: cb.before_calc(i)
        res += i*i
        sleep(1)
        if cb: cb.after_calc(i, val=res)
    return res

In [31]:
slow_calculation(PrintStepCallback())

About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step


30

In [34]:
class PrintStatusCallback():
    def __init__(self): pass
    def before_calc(self, epoch, **kwargs): print(f"About to start: {epoch}")
    def after_calc (self, epoch, val, **kwargs): print(f"After {epoch},  res: {val}")

In [35]:
slow_calculation(PrintStatusCallback())

About to start: 0
After 0,  res: 0
About to start: 1
After 1,  res: 1
About to start: 2
After 2,  res: 5
About to start: 3
After 3,  res: 14
About to start: 4
After 4,  res: 30


30

## Modifying behavior


In [36]:
class PrintAfterCallback():
    def after_calc (self, epoch, val):
        print(f"After {epoch}: {val}")
        if val>10: return True

In [37]:
def slow_calculation(cb=None):
    res = 0
    for i in range(5):
        if cb and hasattr(cb,'before_calc'): cb.before_calc(i)
        res += i*i
        sleep(1)
        if cb and hasattr(cb,'after_calc'):
            if cb.after_calc(i, res):
                print("stopping early")
                break
    return res

In [38]:
slow_calculation(PrintAfterCallback())

After 0: 0
After 1: 1
After 2: 5
After 3: 14
stopping early


14

In [39]:
class ModifyingCallback():
    def after_calc (self, calc, epoch):
        print(f"After {epoch}: {calc.res}")
        if calc.res>10: return True
        # if calc.res<3: calc.res = calc.res*2  # that is just to make smaller values bigger for no real reason just a demo

What is getattr:


    class Dog():
        def bark(self): print("woof")
        def sit(self):  print("sitting")

    d = Dog()

    # NORMAL way:
    d.bark()         # → woof

    # getattr way:
    getattr(d, 'bark')()    # → woof   SAME THING

    and it can also for Dog.attribute not just functions , where we enter it as
    string for the getattr fn

    getattr(d, 'fly',  None)   # why None? → no fly method → returns None (no crash!)

In [40]:
class SlowCalculator():
    def __init__(self, cb=None): self.cb,self.res = cb,0

    def callback(self, cb_name, *args):
        if not self.cb: return  # to prevent crashing if there was no self.cb
        cb = getattr(self.cb,cb_name, None)
        if cb: return cb(self, *args) # here we pass the SlowCalculator attributes which contain self.res

    def calc(self):
        for i in range(5):
            self.callback('before_calc', i)
            self.res += i*i
            sleep(1)
            if self.callback('after_calc', i):
                print("stopping early")
                break

In [41]:
calculator = SlowCalculator(ModifyingCallback())

In [42]:
calculator.calc()
calculator.res

After 0: 0
After 1: 1
After 2: 6
After 3: 15
stopping early


15

# `__dunder__` thingies

Anything that looks like `__this__` is, in some way, *special*. Python, or some library, can define some functions that they will call at certain documented times. For instance, when your class is setting up a new object, python will call `__init__`. These are defined as part of the python [data model](https://docs.python.org/3/reference/datamodel.html#object.__init__).

For instance, if python sees `+`, then it will call the special method `__add__`. If you try to display an object in Jupyter (or lots of other places in Python) it will call `__repr__`.

In [46]:
class SloppyAdder():
    def __init__(self,o): self.o=o
    def __add__(self,b): return SloppyAdder(self.o + b.o + 0.01)
    def __repr__(self): return str(self.o)

In [48]:
a = SloppyAdder(1)
b = SloppyAdder(2)
a+b

3.01

CHEAT SHEET:
────────────────────────────────────────────
- `__init__ `     object created
- `__repr__  `    object shown in console
- `__str__ `      object printed
- `__len__    `   len(object)
- `__getitem__ `  object[index]
- ` __getattr__ `  object.thing  (doesnt exist)
- ` __setattr__  ` object.thing = value
- ` __del__   `    del object
- `__new__ `      before __init__, raw creation
- `__enter__ `    with object:  start
- `__exit__  `    with object:  end (auto cleanup)

────────────────────────────────────────────

    # __repr__  → what shows in the console when you type the object
    # __str__   → what shows when you print() the object

    class Dog():
        def __repr__(self): return f"Dog object"
        def __str__(self):  return f"a dog named {self.name}"

    d = Dog()
    d              # → Dog object       (__repr__)
    print(d)       # → a dog named Rex  (__str__)

-

    # __getattr__  → runs when you access an attribute that DOESNT EXIST
    class Dog():
        def __getattr__(self, name):
            return f"i dont have {name}"

    d = Dog()
    d.bark    # → "i dont have bark"   (doesnt crash!)

-

    # __enter__ and __exit__  → used with the "with" statement
    class File():
        def __enter__(self):
            print("opening file")
            return self

        def __exit__(self, *args):
            print("closing file")  # runs automatically even if crash!

    with File() as f:    # → __enter__ runs
        print("using file")
                         # → __exit__  runs automatically here

-

    # __new__  → runs BEFORE __init__, actually CREATES the object

-

    # __del__  → runs when object is DELETED / garbage collected
    class Dog():
        def __del__(self): print("dog is gone!")

    d = Dog()
    del d    # → "dog is gone!"

##  `__getattr__` and `getattr`

In [49]:
class A: a,b=1,2

In [52]:
x= A()

In [53]:
x.b

2

In [55]:
getattr(x, 'b')

2

In [56]:
getattr(x, 'b' if random.random()>0.5 else 'a') # this is a random one

1

In [57]:
class B:
    a,b=1,2
    def __getattr__(self, k):  # that only run if the attr -> DOESN'T EXIST
        if k[0]=='_': raise AttributeError(k) # that is for the private attributes
        return f'Hello from {k}'

In [58]:
y = B()

In [60]:
y.a

1

In [62]:
y.foo  # here we can see this is something that doesn't exist

'Hello from foo'